# 03. Optional: full fine-tune on a free GPU (Kaggle or Colab)

The shipped model freezes both backbones and trains only the fusion head, which runs on any CPU. This notebook shows the next step: unfreezing the SigLIP backbone and fine-tuning it end to end on the same crops. It is **not executed here** (no GPU in the build environment). On a free Kaggle T4 it takes about 20 minutes.

**Kaggle setup:** New Notebook, Settings, Accelerator: GPU T4 x2, Internet: on. Then run the cells.

In [ ]:
!git clone https://github.com/MoamenFouad/deep-fake-ai-powered-detector.git
%cd deep-fake-ai-powered-detector
!pip -q install -r ml/requirements.txt
!python ml/scripts/prepare_data.py --out data/crops

In [ ]:
import torch, pandas as pd, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification
MODEL = 'prithivMLmods/deepfake-detector-model-v1'  # the SigLIP backbone (explicit fake/real labels)
proc = AutoImageProcessor.from_pretrained(MODEL)
model = AutoModelForImageClassification.from_pretrained(MODEL).cuda()
fake_idx = [i for i, l in model.config.id2label.items() if 'fake' in l.lower()][0]
print(model.config.id2label, 'fake index', fake_idx)

In [ ]:
df = pd.read_csv('data/crops/manifest.csv')

class Crops(Dataset):
    def __init__(self, frame, train):
        self.f, self.train = frame.reset_index(drop=True), train
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        img = Image.open('data/crops/' + self.f.path[i]).convert('RGB')
        if self.train and np.random.rand() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        x = proc(images=img, return_tensors='pt')['pixel_values'][0]
        y = fake_idx if self.f.label[i] == 1 else 1 - fake_idx
        return x, y

train_dl = DataLoader(Crops(df[df.split == 'train'], True), batch_size=32, shuffle=True, num_workers=2)
test_dl = DataLoader(Crops(df[df.split == 'test'], False), batch_size=64, num_workers=2)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5, total_steps=3 * len(train_dl))
scaler = torch.amp.GradScaler()
for epoch in range(3):
    model.train()
    for x, y in train_dl:
        with torch.autocast('cuda', dtype=torch.float16):
            loss = model(pixel_values=x.cuda(), labels=y.cuda()).loss
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    print('epoch', epoch, 'loss', round(loss.item(), 4))

In [ ]:
from sklearn.metrics import roc_auc_score
model.eval(); ps, ys = [], []
with torch.no_grad():
    for x, y in test_dl:
        ps += model(pixel_values=x.cuda()).logits.softmax(-1)[:, fake_idx].cpu().tolist(); ys += (y == fake_idx).int().tolist()
print('test AUC', roc_auc_score(ys, ps))

To use the result: `model.push_to_hub('<your-username>/deepshield-vit')`, set `DS_MODELS` to `buildborderless/CommunityForensics-DeepfakeDet-ViT` plus it backbone, and re-run `ml/scripts/train_fusion.py --candidates ...` so the fusion head matches the new backbone.